# Books to Scrape: catalogue extraction

This notebook scrapes the first five pages of the **All products** catalogue from Books to Scrape. The five pages contain at least 60 books, and each book record includes its title, listed GBP price, star rating, availability text, and category.

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/"
CATALOGUE_URL = urljoin(BASE_URL, "catalogue/")
PAGES_TO_SCRAPE = 5
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; BooksToScrape student project)"
}


def get_soup(url):
    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def get_category(book_url):
    detail_soup = get_soup(book_url)
    breadcrumb = detail_soup.select(".breadcrumb li")
    return breadcrumb[2].get_text(strip=True) if len(breadcrumb) >= 3 else None


rating_names = {"One", "Two", "Three", "Four", "Five"}
all_books = []

for page_number in range(1, PAGES_TO_SCRAPE + 1):
    page_url = (
        urljoin(BASE_URL, "index.html")
        if page_number == 1
        else urljoin(CATALOGUE_URL, f"page-{page_number}.html")
    )
    page_soup = get_soup(page_url)

    for book in page_soup.select("article.product_pod"):
        title_tag = book.select_one("h3 a")
        rating_tag = book.select_one(".star-rating")
        availability_tag = book.select_one(".availability")
        price_tag = book.select_one(".price_color")

        book_url = urljoin(page_url, title_tag["href"])
        rating_classes = rating_tag.get("class", []) if rating_tag else []
        star_rating = next(
            (name for name in rating_names if name in rating_classes),
            None,
        )

        all_books.append(
            {
                "title": title_tag.get("title") if title_tag else None,
                "price": price_tag.get_text(strip=True) if price_tag else None,
                "star_rating": star_rating,
                "availability": (
                    availability_tag.get_text(" ", strip=True)
                    if availability_tag
                    else None
                ),
                "category": get_category(book_url),
            }
        )

books_df = pd.DataFrame(all_books)
required_columns = {
    "title", "price", "star_rating", "availability", "category"
}
assert len(books_df) >= 60, f"Expected at least 60 books, got {len(books_df)}"
assert required_columns.issubset(books_df.columns)
assert books_df[list(required_columns)].notna().all().all()

print(f"Scraped {len(books_df)} books across {PAGES_TO_SCRAPE} catalogue pages.")
print(f"Categories found: {books_df['category'].nunique()}")
books_df.head(10)

ModuleNotFoundError: No module named 'requests'